In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import os
import sys
import time
import numpy as np
import uproot
import math
import itertools
import pandas as pd
import matplotlib.pyplot as plt


from scipy.optimize import Bounds, minimize
from itertools import product

In [ ]:
#
# From: garfieldpp_nitpc/Examples/NITPC/data

## Difusion
SF6_Diffusion = {
    'Field': np.linspace(100, 2000, 20),
    'Long': np.array([
        0.0269729, 0.0190727, 0.0155728, 0.0134865, 0.0120627,
        0.0110116, 0.0101948, 0.00953637, 0.00899097, 0.00852958,
        0.00813264, 0.00778641, 0.00748094, 0.00720881, 0.00696438,
        0.00674323, 0.00654189, 0.00635758, 0.00618801, 0.00603133
    ]),
    'Trans': np.array([
        0.0157342, 0.0111258, 0.00908414, 0.0078671, 0.00703655,
        0.00642346, 0.00594697, 0.00556288, 0.00524473, 0.00497559,
        0.00474404, 0.00454207, 0.00436388, 0.00420514, 0.00406255,
        0.00393355, 0.0038161, 0.00370859, 0.00360967, 0.00351827
    ])
}

## Ion Mobilities
Ar_Mobility = {
    'ReducedField': np.array([
        0, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60, 80, 100,
        120, 150, 200, 250, 300, 400, 500, 600, 800, 1000,
        1200, 1500, 2000
    ]),
    'Mobility': np.array([
        1.53, 1.53, 1.53, 1.53, 1.52, 1.51, 1.49, 1.47, 1.44, 1.41,
        1.38, 1.32, 1.27, 1.22, 1.16, 1.06, 0.99, 0.95, 0.85, 0.78,
        0.72, 0.63, 0.56, 0.51, 0.46, 0.40
    ])
}

CF4_Mobility = {
    'ReducedField': np.array([
        30, 35, 40, 50, 60, 70, 80, 100, 120, 150, 200,
        250, 300, 400, 500, 600, 700, 800, 1000
    ]),
    'Mobility': np.array([
        0.944, 0.944, 0.933, 0.923, 0.924, 0.941, 0.958, 1.026, 1.079,
        1.171, 1.206, 1.220, 1.206, 1.124, 1.036, 0.911, 0.772, 0.772, 0.772
    ])
}

SF6_Mobility = {
    'ReducedField': np.array([
        15, 27, 40, 52, 65, 79, 92, 104, 116, 128, 140,
        152, 200, 300, 400, 500, 600, 800, 1000, 1200, 1500, 2000
    ]),
    'Mobility': np.array([
        0.54, 0.542, 0.54, 0.545, 0.545, 0.548, 0.55, 0.555, 0.56,
        0.568, 0.575, 0.58, 0.6, 0.62, 0.615, 0.605, 0.59, 0.57,
        0.53, 0.53, 0.53, 0.53
    ])
}

## Detatchments
F_e = {
    'Energy': np.array([
        5.61, 6.33, 7.99, 8.56, 12.1, 17.1, 21.9, 28.3, 32.8, 40.1,
        57.1, 59.2, 68.4, 68.6, 104, 145, 186, 226, 315
    ]),
    'CrossSection': np.array([
        0.256, 0.309, 0.196, 0.362, 0.93, 2.4, 3.86, 5.22, 6.88, 8.64,
        9.41, 9.37, 10.1, 10.6, 11.4, 13.6, 14.2, 14.6, 14.8
    ])
}
SF5_F = {
    'Energy': np.array([
        8.47, 10.3, 14.2, 16.2, 18.7, 21.2, 23.4, 27.1, 33.8,
        43.4, 49.2, 59.1, 74.1, 107, 118, 141, 159, 176, 194
    ]),
    'CrossSection': np.array([
        26.1, 29.1, 34.1, 34.8, 35.4, 35.2, 35.2, 35.7, 36.3,
        35.6, 36.0, 35.9, 36.9, 35.5, 33.6, 31.6, 30.5, 29.7, 28.0
    ])
}
SF5_e = {
    'Energy': np.array([
        91.3, 117.0, 135.0, 150.0, 179.0, 201.0, 224.0, 249.0
    ]),
    'CrossSection': np.array([
        0.527, 1.39, 2.42, 3.77, 7.39, 10.8, 13.8, 16.5
    ])
}
SF6_F = {
    'Energy': np.array([
        9.04, 12.0, 15.0, 19.0, 23.4, 32.4, 38.7, 51.8, 71.8,
        91.8, 110.0, 130.0, 150.0, 166.0, 184.0
    ]),
    'CrossSection': np.array([
        21.9, 24.5, 26.4, 27.4, 29.2, 30.0, 30.6, 33.2, 35.4,
        35.6, 35.5, 32.7, 30.3, 29.4, 27.9
    ])
}
SF6_e = {
    'Energy': np.array([
        13.6, 23.9, 47.8, 62.7, 88.7, 113.0, 139.0, 161.0, 185.0,
        210.0, 234.0
    ]),
    'CrossSection': np.array([
        0.256, 0.135, 0.171, 0.157, 0.252, 0.483, 1.5, 3.34, 5.91,
        8.47, 10.8
    ])
}

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

datasets = [
    (Ar_Mobility, 'Ar'),
    (CF4_Mobility, r'CF$_4$'),
    (SF6_Mobility, r'SF$_6$')
]

for inData, label in datasets:
    ax.scatter(
        inData['ReducedField'], inData['Mobility'],
        label=label
    )

ax.set_xlabel(r'Reduced Field (Td = 10$^{-17}$ V cm$^2$)', fontsize=14)
ax.set_ylabel(r'Ion Mobility (cm$^2$ / V / s / atm)', fontsize=14)

ax.set_xscale('log')
ax.grid()
ax.legend(fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

datasets = [
    (F_e, r'SF$_6$ + F$^-$ $\rightarrow$ e$^-$', 'o'),
    (SF5_e, r'SF$_6$ + SF$_5^-$ $\rightarrow$ e$^-$', 'o'),
    (SF6_e, r'SF$_6$ + SF$_6^-$ $\rightarrow$ e$^-$', 'o'),
    (SF5_F, r'SF$_6$ + SF$_5^-$ $\rightarrow$ F$^-$', '^'),
    (SF6_F, r'SF$_6$ + SF$_6^-$ $\rightarrow$ F$^-$', '^')
]

for inData, label, marker in datasets:
    
    ax.scatter(
        inData['Energy'], inData['CrossSection'],
        label=label, marker=marker
    )

ax.scatter(
    [9.04, 12.0, 15.0, 19.0, 23.4, 32.4, 38.7, 51.8, 71.8, 91.8, 110.0, 130.0, 150.0, 166.0, 184.0],
    [21.9, 24.5, 26.4, 27.4, 29.2, 30.0, 30.6, 33.2, 35.4, 35.6, 35.5, 32.7, 30.3, 29.4, 27.9],
    c='c', marker='x', label=r'Simulation (F$^-$)'
)

ax.set_xlabel('Energy (eV)', fontsize=14)
ax.set_ylabel(r'Cross Section ($10^{-20}~\text{m}^2$)', fontsize=14)

ax.set_xscale('log')

ax.grid()
ax.legend(fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
##Ion Masses (TODO - Confirm)
amu = 1.660539e-27 #kg

massSF6 = 146.06*amu

negIonMasses = {
    'F': 19.00*amu,
    'SF5': 127.06*amu,
    'SF6': 146.06*amu
}

In [ ]:
def getDetatchxSec(particleEnergy, energyData, xSecData):
    if len(energyData) == 0:
        raise ValueError('Missing dat.')
        
    sortedIDx = np.argsort(energyData)
    energy = energyData[sortedIDx]
    xSec = xSecData[sortedIDx]
    
    interpData = np.interp(particleEnergy, energy, xSec, left=0.0, right=0.0)
    
    return interpData

In [ ]:
press = 10132.5 #Pa (76 torr)
kb = 1.38e-23 #J/K
temp = 293.15 #K
q = 1.602176e-19

n = press/(kb*temp) #1/m^3
p_atm = press/101325 #atm

eFields = np.logspace(0, 4, 101) #kV/cm
eFields = eFields * 1e5 #V/m

driftLength = 1e-6 # 1um


reducedFields = eFields / n * 1e21
mu_atm = np.interp(reducedFields, SF6_Mobility['ReducedField'], SF6_Mobility['Mobility'])*1e-4
mu_actual = mu_atm / p_atm

vDrift = mu_actual * eFields

In [ ]:
channels = {
    r'SF$_6$ + F$^-$ $\rightarrow$ e$^-$': {'ion': 'F', 'data': F_e},
    r'SF$_6$ + SF$_5^-$ $\rightarrow$ e$^-$': {'ion': 'SF5', 'data': SF5_e},
    r'SF$_6$ + SF$_6^-$ $\rightarrow$ e$^-$': {'ion': 'SF6', 'data': SF6_e},
    r'SF$_6$ + SF$_5^-$ $\rightarrow$ F$^-$': {'ion': 'SF5', 'data': SF5_F},
    r'SF$_6$ + SF$_6^-$ $\rightarrow$ F$^-$': {'ion': 'SF6', 'data': SF6_F},
}

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for label, inChannel in channels.items():
    ionMass = negIonMasses[inChannel['ion']]
    data = inChannel['data']
    
    eKin_J = 1.5 * kb * temp + 0.5 * (ionMass + massSF6) * (vDrift ** 2)
    eKin_eV = eKin_J / q

    xSec = getDetatchxSec(eKin_eV, data['Energy'], data['CrossSection']) * 1e-20
    prob = 1.0 - np.exp(-n * xSec * driftLength)

    ax.plot(eFields / 1e5, prob, label=label, linewidth=2)

ax.plot([1, 40, 40, 3e3], [0, 0, 1, 1], c='r', lw=3, label='Threshold Model')

ax.set_xscale('log')
ax.set_xlabel('Electric Field Strength (kV/cm)', fontsize=14)
ax.set_ylabel(rf'Detachment Probability (over {driftLength * 1e6:.1f} $\mu$m)', fontsize=14)

ax.set_xlim([None, 4e3])

ax.grid()
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import re
import pandas as pd

def parse_garfield_log(file_path):
    electron_pat = re.compile(r"Drifting electron\s+(\d+)")
    field_pat    = re.compile(r"Field \[V/cm\] at \(([^)]+)\)")
    attach_pat   = re.compile(r"Electron Attached at\s+([^\n\r]+)")
    detach_pat   = re.compile(r"Electron Detached at\s+([^\n\r]+)")
    ionise_pat   = re.compile(r"Electron ionised at\s+([^\n\r]+)")
    left_pat     = re.compile(r"Electron left the drift medium at\s+([^\n\r]+)")
    
    trajectory_list, attach_list, detach_list, ionise_list, left_list = [], [], [], [], []
    current_electron = None
    
    with open(file_path, 'r') as f:
        for line in f:
            if match := electron_pat.search(line):
                current_electron = int(match.group(1))
                continue
            
            if current_electron is None:
                continue

            if match := field_pat.search(line):
                coords = [float(v) for v in match.group(1).split(',')]
                trajectory_list.append([current_electron] + coords)
                
            elif match := attach_pat.search(line):
                coords = [float(v) for v in match.group(1).split(',')]
                attach_list.append([current_electron] + coords)
                trajectory_list.append([current_electron] + coords)  # Appended to trajectory
                
            elif match := detach_pat.search(line):
                coords = [float(v) for v in match.group(1).split(',')]
                detach_list.append([current_electron] + coords)
                trajectory_list.append([current_electron] + coords)  # Appended to trajectory
                
            elif match := ionise_pat.search(line):
                coords = [float(v) for v in match.group(1).split(',')]
                ionise_list.append([current_electron] + coords)
                trajectory_list.append([current_electron] + coords)  # Appended to trajectory
                
            elif match := left_pat.search(line):
                coords = [float(v) for v in match.group(1).split(',')]
                left_list.append([current_electron] + coords)
                trajectory_list.append([current_electron] + coords)  # Appended to trajectory
                
    cols = ['electron_id', 'x', 'y', 'z']
    dfs = {
        'data': pd.DataFrame(trajectory_list, columns=cols),
        'attachments': pd.DataFrame(attach_list, columns=cols),
        'detachments': pd.DataFrame(detach_list, columns=cols),
        'ionisations': pd.DataFrame(ionise_list, columns=cols),
        'left_medium': pd.DataFrame(left_list, columns=cols),
    }

    for df in dfs.values():
        if not df.empty:
            df[['x', 'y', 'z']] *= 1e4
            
    return dfs['data'], dfs['attachments'], dfs['detachments'], dfs['ionisations'], dfs['left_medium']

In [ ]:
data, attachments, detachments, ionisations, left_medium = parse_garfield_log('../../../../Downloads/output')

pitch = 55
holeRadius = 17.5
padLength = 20

plotLim = .75*np.array([-pitch, pitch])


fig2D = plt.figure(figsize=(14, 7))
xz = fig2D.add_subplot(221)
yz = fig2D.add_subplot(223)
xy = fig2D.add_subplot(122)

initial = pd.DataFrame({'x': [0], 'y': [0], 'z': [50 * 1e4]})

mapping = [
    (xz, 'x', 'z'),
    (yz, 'y', 'z'),
    (xy, 'x', 'y')
]

for ax, x, y in mapping:

    for ID, track in data.groupby('electron_id'):
        label = 'Drift' if ID == 0 else ''
        ax.plot(track[x], track[y], label=label, lw=1)
        
    ax.scatter(data[x].iloc[0], data[y].iloc[0], marker='^', s=100, c='b', label='Initial')
    ax.scatter(attachments[x], attachments[y], marker='x', s=100, c='g', label=f'Attachment ({len(attachments)})')
    ax.scatter(detachments[x], detachments[y], marker='o', s=75, c='g', label=f'Detachment ({len(detachments)})')
    ax.scatter(ionisations[x], ionisations[y], marker='o', s=25, c='r', label=f'Ionization ({len(ionisations)})')
    ax.scatter(left_medium[x], left_medium[y], marker='x', s=25, c='m', label=f'Exit ({len(left_medium)})')
    
    ax.set_xlabel(rf'{x} ($\mu$m)', fontsize=14)
    ax.set_ylabel(rf'{y} ($\mu$m)', fontsize=14)

for ax in [xz, yz]:
    ax.plot([-pitch, -holeRadius], [0,0], c='k')
    ax.plot([pitch, holeRadius], [0,0], c='k')
    ax.plot([-padLength, padLength], [-50,-50], c='m')
    ax.set_xlim(plotLim)
    ax.set_ylim([-51, data['z'].iloc[0]+5])

hole = plt.Circle(
    (0, 0), holeRadius,
    facecolor='none', edgecolor='k', lw=1
)
xy.add_patch(hole)

padX = padLength*np.array([1., .5, -.5, -1., -.5, .5, 1.])
padY = padLength*math.sqrt(3.)/2.*np.array([0., 1., 1., 0., -1., -1., 0.])
xy.plot(padX, padY, c='m', lw=1)    


xy.set_xlim(plotLim)
xy.set_ylim(plotLim)
xy.legend(fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
import uproot
import matplotlib.pyplot as plt
import numpy as np

# Load ROOT file tree into NumPy arrays
filePath = './../Data/NIDData.root'
with uproot.open(filePath) as file:
    tree = file['particleDataTree']
    allData = tree.arrays(library='np')

for key in ['xPos', 'yPos', 'zPos']:
    allData[key] = allData[key] * 1e4

In [ ]:
allData = pd.DataFrame(allData)
allData

In [ ]:

dataGroup = allData.groupby('avalancheID')
data = dataGroup['avalancheID'==0]

fig2D = plt.figure(figsize=(14, 7))
xz = fig2D.add_subplot(221)
yz = fig2D.add_subplot(223)
xy = fig2D.add_subplot(122)

mapping = [
    (xz, 'xPos', 'zPos'),
    (yz, 'yPos', 'zPos'),
    (xy, 'xPos', 'yPos')
]

numParticles = len(data['numSteps'])

for ax, x, y in mapping:
    for i in range(numParticles):
        ax.plot(data[x][i], data[y][i])

    ax.set_xlabel(rf'{x} ($\mu$m)', fontsize=14)
    ax.set_ylabel(rf'{y} ($\mu$m)', fontsize=14)

for ax in [xz, yz]:
    ax.plot([-pitch, -holeRadius], [0,0], c='k')
    ax.plot([pitch, holeRadius], [0,0], c='k')
    ax.plot([-padLength, padLength], [-50,-50], c='m')
    ax.set_xlim(plotLim)
    ax.set_ylim([-51, 25])

hole = plt.Circle(
    (0, 0), holeRadius,
    facecolor='none', edgecolor='k', lw=1
)
xy.add_patch(hole)

padX = padLength*np.array([1., .5, -.5, -1., -.5, .5, 1.])
padY = padLength*math.sqrt(3.)/2.*np.array([0., 1., 1., 0., -1., -1., 0.])
xy.plot(padX, padY, c='m', lw=1)    

xy.set_xlim(plotLim)
xy.set_ylim(plotLim)

plt.tight_layout()
plt.show()